# SQL Analysis

In [1]:
import sqlite3
import pandas as pd

In [2]:
data=pd.read_csv("../Datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [3]:
data.shape

(7043, 21)

In [4]:
print(data.dtypes)

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


In [5]:
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"],errors="coerce")

In [6]:
print(data["TotalCharges"].dtype)
print(data["TotalCharges"].isna().sum())

float64
11


#### sqlite database connection


In [10]:
conn = sqlite3.connect("../Datasets/processed/customer_intelligence.db")
print("SQLite database connection successful")

SQLite database connection successful


In [11]:
data.to_sql("customers",conn,if_exists="replace",index=False)
print("Customers table created successfully")

Customers table created successfully


#### 1.Total Customers

In [14]:
query = """SELECT COUNT(*) AS total_customers FROM customers;"""
result = pd.read_sql_query(query, conn)
result

,total_customers
0,7043


#### 2. Total churned customers

In [15]:
query = """SELECT COUNT(*) AS total_churned_customers
            FROM customers WHERE Churn = 'Yes';"""

result = pd.read_sql_query(query, conn)
result

,total_churned_customers
0,1869


#### 3. Churn rate

In [16]:
query = """SELECT ROUND(
            COUNT(CASE WHEN Churn = 'Yes' THEN 1 END) * 100.0 / COUNT(*),2
            ) AS churn_rate FROM customers;"""
result = pd.read_sql_query(query, conn)
result

,churn_rate
0,26.54


#### 4. Churn by contract type 

In [17]:
query = """SELECT Contract, Churn, COUNT(*) AS customer_count
         FROM customers GROUP BY Contract, Churn ORDER BY Contract;"""
result = pd.read_sql_query(query, conn)
result

,Contract,Churn,customer_count
0,Month-to-month,No,2220
1,Month-to-month,Yes,1655
2,One year,No,1307
3,One year,Yes,166
4,Two year,No,1647
5,Two year,Yes,48


#### 5. Churn by Payment Method.

In [18]:
query = """SELECT PaymentMethod, Churn, COUNT(*) AS customer_count
         FROM customers
         GROUP BY PaymentMethod, Churn
         ORDER BY PaymentMethod;"""
result = pd.read_sql_query(query, conn)
result

,PaymentMethod,Churn,customer_count
0,Bank transfer (automatic),No,1286
1,Bank transfer (automatic),Yes,258
2,Credit card (automatic),No,1290
3,Credit card (automatic),Yes,232
4,Electronic check,No,1294
5,Electronic check,Yes,1071
6,Mailed check,No,1304
7,Mailed check,Yes,308


#### 6. Churn by Internet Service.

In [19]:
query = """
        SELECT InternetService, Churn, COUNT(*) AS customer_count
        FROM customers
        GROUP BY InternetService, Churn
        ORDER BY InternetService;"""
result = pd.read_sql_query(query, conn)
result

,InternetService,Churn,customer_count
0,DSL,No,1962
1,DSL,Yes,459
2,Fiber optic,No,1799
3,Fiber optic,Yes,1297
4,No,No,1413
5,No,Yes,113


#### 7.Average Monthly Charges.

In [20]:
query = """SELECT ROUND(AVG(MonthlyCharges), 2) AS average_monthly_charges 
        FROM customers;"""
result = pd.read_sql_query(query, conn)
result

,average_monthly_charges
0,64.76


#### 8. Average Total Charges

In [21]:
query = """SELECT ROUND(AVG(TotalCharges), 2) AS average_total_charges
        FROM customers;"""
result = pd.read_sql_query(query, conn)
result

,average_total_charges
0,2283.3


#### 9. High-Value Customers

In [22]:
threshold = data["TotalCharges"].quantile(0.75)
print(threshold)

3794.7375


In [23]:
query = f"""SELECT customerID, tenure, MonthlyCharges, TotalCharges, Contract, Churn
        FROM customers
        WHERE TotalCharges >= {threshold}
        ORDER BY TotalCharges DESC;"""
result = pd.read_sql_query(query, conn)
result.head(10)

,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
0,2889-FPWRM,72,117.80,8684.80,One year,Yes
1,7569-NMZYQ,72,118.75,8672.45,Two year,No
2,9739-JLPQJ,72,117.50,8670.10,Two year,No
3,9788-HNGUT,72,116.95,8594.40,Two year,No
4,8879-XUAHX,71,116.25,8564.75,Two year,No
5,9924-JPRMC,72,118.20,8547.15,Two year,No
6,0675-NCDYU,72,116.40,8543.25,Two year,No
7,6650-BWFRT,72,117.15,8529.50,Two year,No
8,0164-APGRB,72,114.90,8496.70,Two year,No
9,1488-PBLJN,72,116.85,8477.70,Two year,No


#### 10. High-risk customer groups

In [24]:
query = """
SELECT
    Contract,
    COUNT(*) AS customers,
    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS churn_rate
FROM customers
GROUP BY Contract
ORDER BY churn_rate DESC;
"""
result = pd.read_sql_query(query, conn)
result

,Contract,customers,churn_rate
0,Month-to-month,3875,42.71
1,One year,1473,11.27
2,Two year,1695,2.83


#### 11. Customers with long tenure

In [25]:
long_tenure = 48

In [26]:
query = """
SELECT customerID, tenure, Contract, MonthlyCharges, Churn
FROM customers
WHERE tenure >= 48
ORDER BY tenure DESC;
"""
result = pd.read_sql_query(query, conn)
result.head(10)

,customerID,tenure,Contract,MonthlyCharges,Churn
0,5248-YGIJN,72,Two year,90.25,No
1,6234-RAAPL,72,Two year,99.90,No
2,5954-BDFSG,72,Two year,107.50,No
3,0526-SXDJP,72,Two year,42.10,No
4,9848-JQJTX,72,Two year,100.90,No
5,6728-DKUCO,72,One year,104.15,No
6,2848-YXSMW,72,Two year,19.40,No
7,6734-PSBAW,72,Two year,23.55,No
8,3146-MSEGF,72,Two year,88.05,No
9,5997-OPVFA,72,Two year,89.05,No


#### 12. Customers with High Monthly Charges

In [27]:
high_charge = data["MonthlyCharges"].quantile(0.75)
print("High monthly charge threshold:", high_charge)

High monthly charge threshold: 89.85


In [28]:
query = f"""
SELECT customerID, tenure, MonthlyCharges, Contract, InternetService, Churn
FROM customers
WHERE MonthlyCharges >= {high_charge}
ORDER BY MonthlyCharges DESC;
"""
result = pd.read_sql_query(query, conn)
result.head(10)

,customerID,tenure,MonthlyCharges,Contract,InternetService,Churn
0,7569-NMZYQ,72,118.75,Two year,Fiber optic,No
1,8984-HPEMB,71,118.65,Two year,Fiber optic,No
2,5989-AXPUC,68,118.60,Two year,Fiber optic,No
3,5734-EJKXG,61,118.60,One year,Fiber optic,No
4,8199-ZLLSA,67,118.35,One year,Fiber optic,Yes
5,9924-JPRMC,72,118.20,Two year,Fiber optic,No
6,2889-FPWRM,72,117.80,One year,Fiber optic,Yes
7,3810-DVDQQ,72,117.60,Two year,Fiber optic,No
8,9739-JLPQJ,72,117.50,Two year,Fiber optic,No
9,2302-ANTDP,48,117.45,Month-to-month,Fiber optic,Yes


#### 13. Customer segments based on services

##### Count subscribed services

In [29]:
query = """
SELECT
    customerID,
    (
        CASE WHEN PhoneService = 'Yes' THEN 1 ELSE 0 END +
        CASE WHEN MultipleLines = 'Yes' THEN 1 ELSE 0 END +
        CASE WHEN InternetService != 'No' THEN 1 ELSE 0 END +
        CASE WHEN OnlineSecurity = 'Yes' THEN 1 ELSE 0 END +
        CASE WHEN OnlineBackup = 'Yes' THEN 1 ELSE 0 END +
        CASE WHEN DeviceProtection = 'Yes' THEN 1 ELSE 0 END +
        CASE WHEN TechSupport = 'Yes' THEN 1 ELSE 0 END +
        CASE WHEN StreamingTV = 'Yes' THEN 1 ELSE 0 END +
        CASE WHEN StreamingMovies = 'Yes' THEN 1 ELSE 0 END
    ) AS service_count,
    Churn
FROM customers;
"""
result = pd.read_sql_query(query, conn)
result.head()

,customerID,service_count,Churn
0,7590-VHVEG,2,No
1,5575-GNVDE,4,No
2,3668-QPYBK,4,Yes
3,7795-CFOCW,4,No
4,9237-HQITU,2,Yes


##### Create service segments

In [30]:
query = """
SELECT
    CASE
        WHEN service_count <= 2 THEN 'Low services'
        WHEN service_count <= 5 THEN 'Medium services'
        ELSE 'High services'
    END AS service_segment,
    COUNT(*) AS customers,
    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS churned
FROM (
    SELECT
        customerID,
        (
            CASE WHEN PhoneService = 'Yes' THEN 1 ELSE 0 END +
            CASE WHEN MultipleLines = 'Yes' THEN 1 ELSE 0 END +
            CASE WHEN InternetService != 'No' THEN 1 ELSE 0 END +
            CASE WHEN OnlineSecurity = 'Yes' THEN 1 ELSE 0 END +
            CASE WHEN OnlineBackup = 'Yes' THEN 1 ELSE 0 END +
            CASE WHEN DeviceProtection = 'Yes' THEN 1 ELSE 0 END +
            CASE WHEN TechSupport = 'Yes' THEN 1 ELSE 0 END +
            CASE WHEN StreamingTV = 'Yes' THEN 1 ELSE 0 END +
            CASE WHEN StreamingMovies = 'Yes' THEN 1 ELSE 0 END
        ) AS service_count,
        Churn
    FROM customers
)
GROUP BY service_segment
ORDER BY service_segment;
"""
result = pd.read_sql_query(query, conn)
result

,service_segment,customers,churned
0,High services,2187,444
1,Low services,2123,404
2,Medium services,2733,1021


#### 14. Revenue-related SQL analysis

##### Total customer charges

In [31]:
query = """
SELECT
    ROUND(SUM(TotalCharges), 2) AS total_customer_charges
FROM customers;
"""
result = pd.read_sql_query(query, conn)
result

,total_customer_charges
0,16056168.7


##### Customer charges by contract

In [32]:
query = """
SELECT
    Contract,
    COUNT(*) AS customers,
    ROUND(SUM(TotalCharges), 2) AS total_customer_charges,
    ROUND(AVG(TotalCharges), 2) AS average_customer_charges
FROM customers
GROUP BY Contract
ORDER BY total_customer_charges DESC;
"""

result = pd.read_sql_query(query, conn)
result

,Contract,customers,total_customer_charges,average_customer_charges
0,Two year,1695,6283253.7,3728.93
1,Month-to-month,3875,5305861.5,1369.25
2,One year,1473,4467053.5,3034.68


##### Charges by churn status

In [33]:
query = """
SELECT
    Churn,
    COUNT(*) AS customers,
    ROUND(SUM(TotalCharges), 2) AS total_customer_charges,
    ROUND(AVG(TotalCharges), 2) AS average_customer_charges
FROM customers
GROUP BY Churn;
"""
result = pd.read_sql_query(query, conn)
result

,Churn,customers,total_customer_charges,average_customer_charges
0,No,5174,13193241.8,2555.34
1,Yes,1869,2862926.9,1531.80


In [34]:
conn.close()